# 05 · Tools：先跑通一次可观察的工具调用

这一节暂时不做意图分流，只验证最小工具闭环：

HumanMessage → Agent → AIMessage(tool call) → ToolNode → ToolMessage → Agent → final

练习使用 TEU 换算作为确定性工具。只有最终数字不算通关，必须看到工具名、参数、调用 ID 和 ToolMessage。


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Annotated, TypedDict, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / ".env.example").exists():
        ROOT = candidate
        break
os.chdir(ROOT)
load_dotenv(ROOT / ".env")

missing_llm = [
    name for name in ("LLM_BASE_URL", "LLM_MODEL")
    if not (os.getenv(name) or "").strip()
]
if missing_llm:
    raise ValueError(
        "真实 LLM 教学路径必须配置 .env，缺少：" + ", ".join(missing_llm)
    )
print("cwd =", ROOT)
print("mode = live LLM required")

## 1. 先把工具当普通函数验收


In [ ]:
TEU_PER_EQUIPMENT = {"20GP": 1, "40GP": 2, "40HQ": 2}


@tool
def calculate_teu(equipment: str, quantity: int) -> int:
    """按课堂约定换算 TEU。支持 20GP、40GP、40HQ。"""
    key = equipment.strip().upper()
    if key not in TEU_PER_EQUIPMENT:
        raise ValueError(f"不支持的箱型: {equipment}")
    if quantity < 0:
        raise ValueError("quantity 不能为负数")
    return TEU_PER_EQUIPMENT[key] * quantity


assert calculate_teu.invoke({"equipment": "40HQ", "quantity": 3}) == 6
assert calculate_teu.invoke({"equipment": "20GP", "quantity": 2}) == 2
print("05 tool unit ok: 3×40HQ=6, 2×20GP=2")


## 2. 把工具接进 LangGraph

Agent 节点只负责提出工具调用或生成最终回答；ToolNode 才负责校验参数、执行函数并把 ToolMessage 追加到 messages。


In [ ]:
class ToolState(TypedDict):
    messages: Annotated[list, add_messages]


def make_llm():
    from langchain_openai import ChatOpenAI

    return ChatOpenAI(
        model=os.environ["LLM_MODEL"],
        api_key=os.getenv("LLM_API_KEY") or "not-required",
        base_url=os.environ["LLM_BASE_URL"],
        temperature=0,
    )


llm_with_tools = make_llm().bind_tools([calculate_teu])


# TODO(training): 把 calculate_teu 工具接入 LangGraph。
# 1. 实现 agent 节点：让绑定工具后的 LLM 读取 messages 并返回 AIMessage。
# 2. 实现 after_agent 条件路由：有 tool_calls 时进入 tools，否则结束。
# 3. 创建 StateGraph，注册 agent 与 ToolNode，并连接工具调用闭环。
# 完成后应得到 builder，供下方 builder.compile() 使用。

graph = builder.compile()

try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("PNG 不可用，打印 Mermaid：", exc)
    print(graph.get_graph().draw_mermaid())

## 3. 运行真实工具闭环


In [ ]:
result = graph.invoke({
    "messages": [HumanMessage(content="请用工具计算 3×40HQ + 2×20GP 的 TEU，并汇总。")]
})
tool_messages = [message for message in result["messages"] if isinstance(message, ToolMessage)]
print("message_types =", [type(message).__name__ for message in result["messages"]])
print("tool_messages =", [(message.name, message.tool_call_id, message.content) for message in tool_messages])
assert len(tool_messages) == 2
assert all(message.name == "calculate_teu" for message in tool_messages)
assert sum(int(str(message.content)) for message in tool_messages) == 8
print("05 tools live path ok")

## 20 分钟双人练习

使用自己的 AI Coding 工具完成：

1. 新增一个只读工具，输入和返回值必须有明确 schema。
2. 把它接入 StateGraph 的 Agent ↔ ToolNode 循环。
3. 保存一条 AIMessage(tool call) 与对应 ToolMessage。
4. 准备一个非法参数失败样例；失败不得伪装成成功回答。

交付：图结构、一次成功 Trace、一次失败证据。20 分钟后随机抽一组展示。


<!-- codex:checklist -->
---

## 练习任务 Checklist

完成后逐项勾选：

- [ ] 把 `calculate_teu` 当普通函数完成成功与非法参数测试。
- [ ] 找到 AIMessage 中的 tool call 名称、参数和调用 ID。
- [ ] 找到对应 ToolMessage，并核对 `tool_call_id`。
- [ ] 证明 Agent → ToolNode → Agent 闭环得到 8 TEU。
- [ ] 新增一个只读工具，并准备成功与失败各一条 Trace。

**交付证据：**工具单测、AIMessage/ToolMessage 对照、stream event 摘要。